In [1]:
import os
import pandas as pd
import random
from pathlib import Path

# ==============================================================================
# 1. 配置路径
# ==============================================================================
PROJECT_ROOT = Path(".").resolve()
BASE_WF    = PROJECT_ROOT / "WaveFake"
BASE_ASV   = PROJECT_ROOT / "LA"
BASE_WILD  = PROJECT_ROOT / "In_The_Wild"
BASE_FOR   = PROJECT_ROOT / "Fake_or_Real"

OUTPUT_DIR = PROJECT_ROOT / "protocol"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def build_protocol():
    master_list = []

    def safe_add(full_path, label, model_name, source):
        p = Path(full_path)
        if p.exists():
            try:
                rel_path = p.relative_to(PROJECT_ROOT)
            except ValueError:
                rel_path = p
            master_list.append({
                "path": str(rel_path),
                "label": label, 
                "model_name": model_name, 
                "source": source
            })
            return True
        return False

    def get_files(path, ext=(".wav", ".WAV", ".flac")):
        if not os.path.exists(path): return []
        return [os.path.join(path, f) for f in os.listdir(path) if f.lower().endswith(ext)]

    print("🚀 Starting Balanced Dataset Generation...")

    # ---- 1. FoR 数据集：核心攻坚 (目标: ~55,000条) ----
    # 策略：Rerecorded 全保，Original/Norm 大比例抽样
    FOR_CONFIG = {
        "for-rerecorded": {"cap_per_label": 6500},  # 尽量全拿 (13k total)
        "for-2seconds":   {"cap_per_label": 7500},  # 强攻短语音 (15k total)
        "for-original":   {"cap_per_label": 7500},  # 基础特征 (15k total)
        "for-norm":       {"cap_per_label": 6000}   # 归一化特征 (12k total)
    }

    for scenario, cfg in FOR_CONFIG.items():
        for label_name in ["real", "fake"]:
            # 合并训练和验证集，统一重新平衡采样
            combined_files = []
            for split in ["training", "validation"]:
                path = BASE_FOR / scenario / split / label_name
                combined_files.extend(get_files(path))
            
            sample_size = min(len(combined_files), cfg["cap_per_label"])
            sampled = random.sample(combined_files, sample_size)
            for f in sampled:
                safe_add(f, 1 if label_name == "real" else 0, scenario, f"FOR_{scenario}")

    # ---- 2. WaveFake：基础指纹 (目标: ~25,000条) ----
    # 策略：减少单一模型冗余
    fake_folders = [
        "ljspeech_full_band_melgan", "ljspeech_hifiGAN", "ljspeech_melgan",
        "ljspeech_melgan_large", "ljspeech_multi_band_melgan",
        "ljspeech_parallel_wavegan", "ljspeech_waveglow"
    ]
    # 每个模型只取 1500 条，防止某一个 Vocoder 霸占特征空间
    for folder in fake_folders:
        f_files = get_files(BASE_WF / folder)
        for f in random.sample(f_files, min(len(f_files), 1500)):
            safe_add(f, 0, folder.split('_')[-1], "WF_LJ_Fake")
            
    # LJ_Real 对应取 10,000 条保持比例
    lj_reals = get_files(BASE_WF / "LJSpeech-1.1_real")
    for f in random.sample(lj_reals, min(len(lj_reals), 10000)):
        safe_add(f, 1, "original", "LJ_Real")

    # ---- 3. WF_CV (Common Voice)：人声多样性 (目标: 10,000条) ----
    cv_files = get_files(BASE_WF / "common_voices_prompts_from_conformer_fastspeech2_pwg_ljspeech")
    for f in random.sample(cv_files, min(len(cv_files), 10000)):
        safe_add(f, 0, "fastspeech2_pwg", "WF_CV")

    # ---- 4. In-The-Wild：野外泛化 (目标: 12,000条) ----
    wild_csv = BASE_WILD / "meta.csv"
    if wild_csv.exists():
        df_wild = pd.read_csv(wild_csv)
        # 1:1 采样真假样本，总量控制在 12000
        reals = df_wild[df_wild['label'] == 'bona-fide']
        fakes = df_wild[df_wild['label'] == 'spoof']
        sample_n = min(len(reals), len(fakes), 6000)
        df_wild_balanced = pd.concat([reals.sample(sample_n), fakes.sample(sample_n)])
        for _, row in df_wild_balanced.iterrows():
            safe_add(BASE_WILD / str(row['file']).strip(), 1 if row['label'] == 'bona-fide' else 0, "wild", "Wild")

    # ---- 5. ASVspoof & JSUT：学术基准 (目标: 15,000条) ----
    # ASVspoof 2019 LA
    asv_prot = BASE_ASV / "ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt"
    if asv_prot.exists():
        asv_list = []
        with open(asv_prot, "r") as f:
            for line in f:
                parts = line.strip().split()
                asv_list.append((parts[1], 1 if parts[4] == "bonafide" else 0, parts[3]))
        # 采样 8000 条
        for name, lbl, model in random.sample(asv_list, min(len(asv_list), 8000)):
            path = BASE_ASV / "ASVspoof2019_LA_train/flac" / f"{name}.flac"
            safe_add(path, lbl, model, "ASV")

    # JSUT
    for f in get_files(BASE_WF / "jsut_real"): # 全部 JSUT 比较小，可以全带上
        safe_add(f, 1, "original", "JSUT")
    for folder in ["jsut_multi_band_melgan", "jsut_parallel_wavegan"]:
        for f in get_files(BASE_WF / folder):
            safe_add(f, 0, "melgan/pwg", "JSUT")

    # ==============================================================================
    # 报告生成
    # ==============================================================================
    df_final = pd.DataFrame(master_list)
    final_path = OUTPUT_DIR / "master_protocol.csv"
    df_final.to_csv(final_path, index=False)

    print("\n" + "="*70)
    print(f"{'INDUSTRIAL BALANCED PROTOCOL REPORT':^70}")
    print("="*70)
    print(f"  Total Samples : {len(df_final)}")
    print(f"  Fake/Real     : {df_final['label'].value_counts().to_dict()}")
    print("-" * 70)
    counts = df_final['source'].value_counts()
    for src, count in counts.items():
        print(f"{src:<35} | {count:<10} | {(count/len(df_final))*100:>8.2f}%")
    print("="*70)
    print(f"✅ Mission Accomplished: {final_path}")

if __name__ == "__main__":
    build_protocol()

🚀 Starting Balanced Dataset Generation...

                 INDUSTRIAL BALANCED PROTOCOL REPORT                  
  Total Samples : 116390
  Fake/Real     : {0: 67376, 1: 49014}
----------------------------------------------------------------------
FOR_for-2seconds                    | 15000      |    12.89%
JSUT                                | 15000      |    12.89%
FOR_for-rerecorded                  | 12452      |    10.70%
FOR_for-norm                        | 12000      |    10.31%
Wild                                | 12000      |    10.31%
FOR_for-original                    | 11438      |     9.83%
WF_LJ_Fake                          | 10500      |     9.02%
LJ_Real                             | 10000      |     8.59%
WF_CV                               | 10000      |     8.59%
ASV                                 | 8000       |     6.87%
✅ Mission Accomplished: /home/jesse/Projects/MyProjects/2026_FinQuest/Finquest_Wav2Vec_Deepfake/protocol/master_protocol.csv
